# Data Cleaning Notebook
Upload a CSV or Excel file, run the cells top to bottom, and download the cleaned file at the end.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import re
from google.colab import files

## 2. Upload your dataset

In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print("Loaded:", file_name)

## 3. Load the data

In [ ]:
if file_name.lower().endswith(".csv"):
    try:
        df = pd.read_csv(file_name)
    except UnicodeDecodeError:
        df = pd.read_csv(file_name, encoding="latin-1")
elif file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name)
else:
    raise ValueError("Please upload a .csv, .xlsx or .xls file")

df_raw = df.copy()   # keep the original for comparison
df.head()

## 4. Quick overview

In [ ]:
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe(include="all").T

## 5. Clean column names

In [ ]:
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
              .str.strip("_")
)
print(list(df.columns))

## 6. Trim whitespace and standardize missing-value markers

In [ ]:
text_cols = df.select_dtypes(include="object").columns

for c in text_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].str.replace(r"\s+", " ", regex=True)

missing_tokens = ["", "nan", "NaN", "NAN", "null", "NULL", "None", "none", "N/A", "n/a", "NA", "na", "-", "--", "?"]
df.replace(missing_tokens, np.nan, inplace=True)
df.isna().sum()

## 7. Remove duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows")

## 8. Fix data types
Edit the lists below with your own column names.

In [ ]:
numeric_cols = []   # e.g. ["age", "salary", "price"]
date_cols    = []   # e.g. ["order_date", "joined_on"]

for c in numeric_cols:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.replace(r"[^0-9.\-]", "", regex=True),
        errors="coerce"
    )

for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

df.dtypes

## 9. Standardize text columns (case, typos)

In [ ]:
text_cols = df.select_dtypes(include="object").columns

for c in text_cols:
    df[c] = df[c].str.title()   # use .str.lower() if you prefer lowercase

# Example of fixing specific inconsistent values:
# df["city"] = df["city"].replace({"Bombay": "Mumbai", "Banglore": "Bengaluru"})

## 10. Handle missing values
Pick the strategy that suits your data.

In [ ]:
# Drop columns that are mostly empty (more than 60% missing)
threshold = 0.6
mostly_empty = df.columns[df.isna().mean() > threshold]
df = df.drop(columns=mostly_empty)
print("Dropped columns:", list(mostly_empty))

# Fill numeric columns with the median
num_cols = df.select_dtypes(include=np.number).columns
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

# Fill text columns with the most common value (or "Unknown")
cat_cols = df.select_dtypes(include="object").columns
for c in cat_cols:
    if df[c].notna().any():
        df[c] = df[c].fillna(df[c].mode()[0])
    else:
        df[c] = df[c].fillna("Unknown")

# Or drop rows that still have missing values:
# df = df.dropna()

df.isna().sum()

## 11. Handle outliers (IQR method)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns

for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[c] < low) | (df[c] > high)).sum()
    print(f"{c}: {n_out} outliers")
    # Cap the values instead of deleting rows:
    df[c] = df[c].clip(lower=low, upper=high)

## 12. Final check

In [ ]:
print("Before:", df_raw.shape, "| After:", df.shape)
print("Missing values left:", df.isna().sum().sum())
print("Duplicates left:", df.duplicated().sum())
df.head()

## 13. Save and download the cleaned file

In [ ]:
df.to_csv("cleaned_data.csv", index=False)
files.download("cleaned_data.csv")